# ITU Synthetic Mobile Phone Data – Dataset Validation

## Project
Telecom Usage Event Platform with Apache Cassandra

## Objective
This notebook performs data profiling and quality validation
on the ITU Synthetic Mobile Phone Dataset (Nairobi, Kenya).

## Dataset Overview
- Source: International Telecommunication Union (ITU)
- Dataset type: Synthetic Mobile Phone Data
- Coverage: Nairobi, Kenya
- Period: February 2026
- Expected subscribers: 3,000

## Validation Scope
- Dataset structure and schema
- Data types and missing values
- Duplicate records
- Timestamp validity
- Geographic coordinate validity
- Subscriber and network distribution

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("/home/sina/bigdata-lab/telecom-usage-cassandra/data/raw/MPD_sample_synthetic_ken_3000subs.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (1072608, 7)


## 1. Dataset Structure and Data Types

This section examines the dataset schema, column names,
inferred data types, and memory usage.

In [4]:
print("===== Column Names =====")
print(df.columns.tolist())

print("\n===== Data Types =====")
print(df.dtypes)

print("\n===== Dataset Information =====")
df.info()

===== Column Names =====
['msisdn', 'datetime', 'cell_id', 'latitude', 'longitude', 'data_type', 'service']

===== Data Types =====
msisdn           str
datetime         str
cell_id        int64
latitude     float64
longitude    float64
data_type        str
service          str
dtype: object

===== Dataset Information =====
<class 'pandas.DataFrame'>
RangeIndex: 1072608 entries, 0 to 1072607
Data columns (total 7 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   msisdn     1072608 non-null  str    
 1   datetime   1072608 non-null  str    
 2   cell_id    1072608 non-null  int64  
 3   latitude   1072608 non-null  float64
 4   longitude  1072608 non-null  float64
 5   data_type  1072608 non-null  str    
 6   service    1072608 non-null  str    
dtypes: float64(2), int64(1), str(4)
memory usage: 99.3 MB


## 2. Missing Values Analysis

This section identifies missing values in each column
and evaluates the completeness of the dataset.

In [5]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (df.isnull().mean() * 100).round(4)
})

missing_summary

,Missing_Count,Missing_Percentage
msisdn,0,0.0
datetime,0,0.0
cell_id,0,0.0
latitude,0,0.0
longitude,0,0.0
data_type,0,0.0
service,0,0.0


## 3. Duplicate Records Analysis

This section identifies fully duplicated rows
across all dataset columns.

In [6]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate records: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_count / len(df) * 100:.4f}%")

Duplicate records: 0
Duplicate percentage: 0.0000%


## 4. Timestamp Validation

This section validates timestamp values, converts the datetime
column to Pandas datetime format, and identifies invalid records.

In [7]:
df["datetime"] = pd.to_datetime(
    df["datetime"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

invalid_datetime = df["datetime"].isna().sum()

print(f"Invalid datetime records: {invalid_datetime:,}")
print(f"Minimum datetime: {df['datetime'].min()}")
print(f"Maximum datetime: {df['datetime'].max()}")
print(f"Datetime dtype: {df['datetime'].dtype}")

Invalid datetime records: 0
Minimum datetime: 2026-02-01 00:02:00
Maximum datetime: 2026-02-28 23:59:00
Datetime dtype: datetime64[us]


## 5. Geographic Coordinate Validation

This section checks whether latitude and longitude values
fall within their valid geographic ranges.

In [8]:
invalid_latitude = ~df["latitude"].between(-90, 90)
invalid_longitude = ~df["longitude"].between(-180, 180)

print(f"Invalid latitude records: {invalid_latitude.sum():,}")
print(f"Invalid longitude records: {invalid_longitude.sum():,}")

print("\n===== Coordinate Ranges =====")
print("Latitude:", df["latitude"].min(), "to", df["latitude"].max())
print("Longitude:", df["longitude"].min(), "to", df["longitude"].max())

Invalid latitude records: 0
Invalid longitude records: 0

===== Coordinate Ranges =====
Latitude: -1.4997 to -1.1003
Longitude: 36.6001 to 37.0998


## 6. Subscriber and Network Cell Analysis

This section examines the number of unique subscribers
and network cells represented in the dataset.

In [10]:
print("===== Subscriber Statistics =====")
print("Unique subscribers:", df["msisdn"].nunique())
print("Total events:", len(df))
print("Average events per subscriber:",
      round(len(df) / df["msisdn"].nunique(), 2))

print("\n===== Network Cell Statistics =====")
print("Unique cells:", df["cell_id"].nunique())

===== Subscriber Statistics =====
Unique subscribers: 3000
Total events: 1072608
Average events per subscriber: 357.54

===== Network Cell Statistics =====
Unique cells: 35956


In [11]:
print("===== Data Type Distribution =====")
print(df["data_type"].value_counts(dropna=False))

print("\n===== Service Distribution =====")
print(df["service"].value_counts(dropna=False))

===== Data Type Distribution =====
data_type
CDR     1009762
IPDR      62846
Name: count, dtype: int64

===== Service Distribution =====
service
2G    1003186
4G      48751
3G      20671
Name: count, dtype: int64
